# Lab 3: Multi-Source Retail Sales Data Integration and Analysis

**Platform:** R / Google Colab  
**Domain:** Retail Analytics

This notebook implements the complete experiment:

1. Import and clean CSV, JSON, and Excel data
2. Integrate multiple data sources
3. Perform sales and customer analysis
4. Store and retrieve the integrated data using SQLite
5. Generate three business insights

> **Note:** Upload the UCI `Online Retail.xlsx` file when prompted. The notebook automatically creates `transactions.csv`, `products.json`, and `customers.xlsx` from the original dataset so the remaining experiment exactly follows the required multi-source workflow.


## 1. Install and Load Required R Packages

The following packages are used:
- `tidyverse` for data manipulation
- `jsonlite` for JSON files
- `readxl` and `writexl` for Excel files
- `DBI` and `RSQLite` for SQLite database operations


In [ ]:
# Install required packages if they are not already installed
packages <- c("tidyverse", "jsonlite", "readxl", "writexl", "DBI", "RSQLite")

installed <- rownames(installed.packages())
for (p in packages) {
  if (!(p %in% installed)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(jsonlite)
library(readxl)
library(writexl)
library(DBI)
library(RSQLite)

cat("All required packages loaded successfully.
")

## 2. Load the UCI Online Retail Dataset

The notebook automatically downloads the official **Online Retail.xlsx** dataset from the UCI Machine Learning Repository if it is not already present. This makes the notebook ready to run directly in Google Colab.

In [ ]:
# Automatically download the official UCI Online Retail dataset if needed

if (!file.exists("Online Retail.xlsx")) {
  dataset_url <- "https://archive.ics.uci.edu/static/public/352/online%2Bretail.zip"
  zip_file <- "online_retail.zip"

  cat("Downloading UCI Online Retail dataset...\n")
  download.file(dataset_url, zip_file, mode = "wb")
  unzip(zip_file)
}

if (!file.exists("Online Retail.xlsx")) {
  stop("Online Retail.xlsx could not be found after download.")
}

cat("Dataset is ready: Online Retail.xlsx\n")

## 3. Read the Original Dataset and Create the Three Required Data Sources

The supplied retail data is organized into:
- `transactions.csv`: InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate
- `products.json`: StockCode, Description, UnitPrice
- `customers.xlsx`: CustomerID, Country


In [ ]:
# Read the official UCI Online Retail Excel file
source_file <- "Online Retail.xlsx"
cat("Using source file:", source_file, "\n")

raw_data <- read_excel(source_file)

cat("Original dataset dimensions:", nrow(raw_data), "rows x", ncol(raw_data), "columns\n")
head(raw_data)

In [ ]:
# Standardize column names if necessary
names(raw_data) <- trimws(names(raw_data))

required_cols <- c("InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country")
missing_required <- setdiff(required_cols, names(raw_data))

if (length(missing_required) > 0) {
  stop(paste("Missing expected columns:", paste(missing_required, collapse = ", ")))
}

# Build transactions.csv
transactions_source <- raw_data %>%
  select(InvoiceNo, StockCode, CustomerID, Quantity, InvoiceDate)
write_csv(transactions_source, "transactions.csv")

# Build products.json
# Keep unique StockCode-Description-UnitPrice combinations.
products_source <- raw_data %>%
  select(StockCode, Description, UnitPrice) %>%
  distinct()
write_json(products_source, "products.json", pretty = TRUE, na = "null")

# Build customers.xlsx
customers_source <- raw_data %>%
  select(CustomerID, Country) %>%
  distinct()
write_xlsx(customers_source, "customers.xlsx")

cat("Created transactions.csv, products.json and customers.xlsx successfully.
")

# Task 1: Import and Clean the Data

In [ ]:
# Import the three heterogeneous data sources
transactions <- read_csv("transactions.csv", show_col_types = FALSE)
products <- fromJSON("products.json") %>% as_tibble()
customers <- read_excel("customers.xlsx")

cat("Transactions dimensions:", dim(transactions), "
")
cat("Products dimensions:", dim(products), "
")
cat("Customers dimensions:", dim(customers), "
")

head(transactions)
head(products)
head(customers)

In [ ]:
# Inspect missing values before cleaning
cat("Missing values in transactions:
")
print(colSums(is.na(transactions)))

cat("
Missing values in products:
")
print(colSums(is.na(products)))

cat("
Missing values in customers:
")
print(colSums(is.na(customers)))

In [ ]:
# Data cleaning

# 1. Remove duplicate transaction records
transactions_clean <- transactions %>%
  distinct()

# 2. Remove missing key identifiers and invalid/zero quantities
transactions_clean <- transactions_clean %>%
  filter(
    !is.na(InvoiceNo),
    !is.na(StockCode),
    !is.na(CustomerID),
    !is.na(Quantity),
    Quantity > 0
  )

# 3. Clean products: remove missing product details and invalid/zero prices
products_clean <- products %>%
  distinct() %>%
  filter(
    !is.na(StockCode),
    !is.na(Description),
    !is.na(UnitPrice),
    UnitPrice > 0
  )

# In the raw UCI data a StockCode can occasionally appear with multiple prices.
# For integration, create one representative product record per StockCode using median UnitPrice
# and the first available non-missing description.
products_clean <- products_clean %>%
  group_by(StockCode) %>%
  summarise(
    Description = first(Description),
    UnitPrice = median(UnitPrice, na.rm = TRUE),
    .groups = "drop"
  )

# 4. Clean customer data
customers_clean <- customers %>%
  distinct() %>%
  filter(!is.na(CustomerID), !is.na(Country)) %>%
  group_by(CustomerID) %>%
  summarise(Country = first(Country), .groups = "drop")

cat("Rows after cleaning:
")
cat("Transactions:", nrow(transactions_clean), "
")
cat("Products:", nrow(products_clean), "
")
cat("Customers:", nrow(customers_clean), "
")

### Cleaning Decisions

- Duplicate records were removed using `distinct()`.
- Transactions with missing InvoiceNo, StockCode, CustomerID, or Quantity were removed.
- Transactions with zero or negative Quantity were removed because the experiment focuses on valid sales transactions.
- Products with missing descriptions/prices or UnitPrice less than or equal to zero were removed.
- One representative product record was retained per StockCode so that joining does not duplicate transactions.
- Customer rows with missing CustomerID or Country were removed.


# Task 2: Integrate the Multiple Data Sources

In [ ]:
# Check unmatched records before joining
unmatched_products <- transactions_clean %>%
  anti_join(products_clean, by = "StockCode")

unmatched_customers <- transactions_clean %>%
  anti_join(customers_clean, by = "CustomerID")

cat("Transactions without matching product:", nrow(unmatched_products), "
")
cat("Transactions without matching customer:", nrow(unmatched_customers), "
")

In [ ]:
# Integrate datasets using INNER JOIN
# Inner join is selected because the analysis requires valid transaction,
# product-price and customer-country information for every row.

retail_sales <- transactions_clean %>%
  inner_join(products_clean, by = "StockCode") %>%
  inner_join(customers_clean, by = "CustomerID") %>%
  mutate(Revenue = Quantity * UnitPrice)

cat("Final integrated dataset dimensions:", nrow(retail_sales), "rows x", ncol(retail_sales), "columns
")
head(retail_sales)

### Join Justification

`inner_join()` is used because a complete analytical record must contain:
- transaction information,
- valid product description and price, and
- customer country.

Rows that do not have a valid match in all required sources are excluded from final revenue analysis. The unmatched-record counts are displayed above for verification.


# Task 3: Sales and Customer Analysis

In [ ]:
# 1. Total sales revenue
total_revenue <- sum(retail_sales$Revenue, na.rm = TRUE)
cat("Total Sales Revenue =", round(total_revenue, 2), "
")

In [ ]:
# 2. Top 5 products based on revenue
top5_products <- retail_sales %>%
  group_by(StockCode, Description) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)

top5_products

In [ ]:
# 3. Top 5 countries based on revenue
top5_countries <- retail_sales %>%
  group_by(Country) %>%
  summarise(TotalRevenue = sum(Revenue, na.rm = TRUE), .groups = "drop") %>%
  arrange(desc(TotalRevenue)) %>%
  slice_head(n = 5)

top5_countries

In [ ]:
# 4. Top 5 customers based on total purchase value
customer_value <- retail_sales %>%
  group_by(CustomerID, Country) %>%
  summarise(TotalPurchaseValue = sum(Revenue, na.rm = TRUE), .groups = "drop")

top5_customers <- customer_value %>%
  arrange(desc(TotalPurchaseValue)) %>%
  slice_head(n = 5)

top5_customers

In [ ]:
# Customer segmentation using data-driven quartile thresholds
q1 <- quantile(customer_value$TotalPurchaseValue, 0.25, na.rm = TRUE)
q2 <- quantile(customer_value$TotalPurchaseValue, 0.50, na.rm = TRUE)
q3 <- quantile(customer_value$TotalPurchaseValue, 0.75, na.rm = TRUE)

customer_segments <- customer_value %>%
  mutate(
    CustomerSegment = case_when(
      TotalPurchaseValue <= q1 ~ "Low Value",
      TotalPurchaseValue <= q2 ~ "Medium Value",
      TotalPurchaseValue <= q3 ~ "High Value",
      TRUE ~ "Premium"
    )
  )

cat("Thresholds used:
")
cat("Q1 =", round(q1, 2), "
")
cat("Median =", round(q2, 2), "
")
cat("Q3 =", round(q3, 2), "

")

customer_segments %>% count(CustomerSegment)

In [ ]:
# Identify high-performing and underperforming markets
country_performance <- retail_sales %>%
  group_by(Country) %>%
  summarise(
    Revenue = sum(Revenue, na.rm = TRUE),
    Orders = n_distinct(InvoiceNo),
    Customers = n_distinct(CustomerID),
    .groups = "drop"
  ) %>%
  arrange(desc(Revenue))

high_market <- country_performance %>% slice_head(n = 1)
low_market <- country_performance %>% slice_tail(n = 1)

cat("High-performing market:
")
print(high_market)

cat("
Underperforming market:
")
print(low_market)

cat("
Observation:
")
cat(high_market$Country, "is the highest-performing market because it generates the maximum revenue in the cleaned dataset.
")
cat(low_market$Country, "is the underperforming market because it generates the lowest revenue among the countries represented after cleaning.
")

# Task 4: Store and Retrieve Data Using SQL

In [ ]:
# Create SQLite database and save final dataset
con <- dbConnect(RSQLite::SQLite(), "retail_analysis.db")

dbWriteTable(con, "retail_sales", retail_sales, overwrite = TRUE)

cat("Table created successfully.
")
print(dbListTables(con))
cat("Rows stored in retail_sales:", dbGetQuery(con, "SELECT COUNT(*) AS n FROM retail_sales")$n, "
")

In [ ]:
# SQL Query 1: Top 5 customers based on revenue
query1 <- "
SELECT CustomerID,
       Country,
       ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM retail_sales
GROUP BY CustomerID, Country
ORDER BY TotalRevenue DESC
LIMIT 5;
"

sql_top_customers <- dbGetQuery(con, query1)
sql_top_customers

In [ ]:
# SQL Query 2: Total revenue by country
query2 <- "
SELECT Country,
       ROUND(SUM(Revenue), 2) AS TotalRevenue
FROM retail_sales
GROUP BY Country
ORDER BY TotalRevenue DESC;
"

sql_country_revenue <- dbGetQuery(con, query2)
head(sql_country_revenue, 10)

# Business Insights

In [ ]:
# Automatically generate three business insights based on notebook results
best_product <- top5_products %>% slice_head(n = 1)
best_country <- top5_countries %>% slice_head(n = 1)
best_customer <- top5_customers %>% slice_head(n = 1)

cat("INSIGHT 1:
")
cat(best_country$Country, "is the strongest market, contributing approximately",
    round(best_country$TotalRevenue, 2), "in revenue. Management should protect and expand this key market.

")

cat("INSIGHT 2:
")
cat("The highest-revenue product is", best_product$Description,
    "with revenue of approximately", round(best_product$TotalRevenue, 2),
    ". This product should receive priority in inventory and promotion decisions.

")

cat("INSIGHT 3:
")
cat("Customer", best_customer$CustomerID, "is the highest-value customer with total purchases of approximately",
    round(best_customer$TotalPurchaseValue, 2),
    ". Premium/high-value customers are suitable targets for retention and loyalty strategies.
")

In [ ]:
# Close database connection
if (dbIsValid(con)) {
  dbDisconnect(con)
}
cat("SQLite connection closed successfully.
")

# Final Conclusion

This experiment successfully demonstrated an end-to-end retail analytics workflow using R. Data from CSV, JSON, and Excel sources were imported, cleaned, integrated, analyzed, and stored in SQLite. The analysis identified total revenue, top products, top countries, high-value customers, customer segments, and market performance. The generated SQLite database can be reused for future analysis and SQL queries.

## Files Produced by This Notebook
- `transactions.csv`
- `products.json`
- `customers.xlsx`
- `retail_analysis.db`

These files, along with this notebook and its PDF export, can be added to the Git repository for submission.
